# Topic 15 check: does the speech-codec pipeline run?

For tutors (and Topic 15 groups on day 1). Runs every part of `speech_kit.py` once, in about 10 minutes:
LibriSpeech download, speaker check, Opus baseline with PESQ/STOI, and a **deliberately tiny** neural codec trained for one minute, from training batches to bitrate, quality and real-time factor.

It downloads only dev-clean and test-clean (about 0.7 GB). The tiny codec stands in for the groups' models: its quality will be poor, that is expected. The point is that every step runs.

For a GPU: *Runtime → Change runtime type → T4 GPU* (the check also runs on a CPU, more slowly).
Upload `starter_kit.zip` to the Files panel first, as for the demo notebook.

In [ ]:
KIT_REPO = ''   # URL of the kit's Git repository: then no upload of starter_kit.zip is needed
import os, sys, subprocess, zipfile, shutil, time
if not os.path.isdir('starter_kit'):
    if KIT_REPO and not os.path.isfile('starter_kit.zip'):
        subprocess.run(['git', 'clone', '-q', KIT_REPO, 'starter_kit'], check=True)
    else:
        if not zipfile.is_zipfile('starter_kit.zip'):
            raise SystemExit('Upload starter_kit.zip first (and wait until the upload has finished).')
        zipfile.ZipFile('starter_kit.zip').extractall('.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'pesq', 'pystoi', 'dahuffman'], check=True)
sys.path.insert(0, os.path.abspath('starter_kit/topic15'))
import numpy as np, speech_kit as s
sys.path.insert(0, os.path.abspath('starter_kit')); import ac_kit
print('kit version', ac_kit.KIT_VERSION, '| python', sys.version.split()[0], '| ffmpeg', bool(shutil.which('ffmpeg')), '| ffprobe', bool(shutil.which('ffprobe')))
results = {}

## 1. LibriSpeech: download, speaker check, segments

In [ ]:
t0 = time.time()
base = s.download_librispeech(subsets=('dev-clean', 'test-clean'), root='LibriSpeech_data')
print('download + unpack: %.0f s' % (time.time() - t0))
print('speakers per split:', s.check_speakers_disjoint(base))
dev, test = s.list_utterances(base, 'dev-clean'), s.list_utterances(base, 'test-clean')
print(len(dev), 'dev utterances,', len(test), 'test utterances')
seg = s.random_segments(dev, 8, seconds=1.0, rng=0)
assert seg.shape == (8, 16000) and np.array_equal(seg, s.random_segments(dev, 8, seconds=1.0, rng=0))
results['librispeech'] = 'ok'

## 2. Opus baseline (restricted low delay, 10 ms frames = 12.5 ms algorithmic delay)

In [ ]:
rows = []
for kbps in (8, 12, 16):
    p, q, r = [], [], []
    for path, _ in test[:5]:
        x = s.read_audio(path)
        o = s.opus_roundtrip(x, kbps=kbps, frame_ms=10)
        p.append(s.pesq_wb(x, o['decoded'])); q.append(s.stoi_score(x, o['decoded'])); r.append(o['kbps'])
    rows.append((kbps, np.mean(r), np.mean(p), np.mean(q)))
    print('Opus %2d kbit/s: measured %.2f kbit/s, PESQ %.2f, STOI %.3f (mean of 5 test utterances)' % rows[-1])
assert rows[0][2] < rows[1][2] < rows[2][2], 'PESQ should rise with bitrate'
results['opus'] = 'ok'

## 3. A tiny neural codec, end to end
Frames of 160 samples (10 ms), no look-ahead: algorithmic delay 10 ms. 8 latent values per frame, each quantized to 16 levels (straight-through rounding), entropy-coded with Huffman. Trained for about one minute on dev-clean only, which is not how a group should train (train-clean-100, much longer).

In [ ]:
import torch, torch.nn as nn
dev_name = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev_name, torch.cuda.get_device_name(0) if dev_name == 'cuda' else '')

class TinyCodec(nn.Module):
    def __init__(self, frame=160, ch=64, latent=8, levels=16):
        super().__init__()
        self.frame, self.levels = frame, levels
        self.enc = nn.Sequential(nn.Conv1d(1, ch, frame, stride=frame), nn.GELU(), nn.Conv1d(ch, latent, 1), nn.Tanh())
        self.dec = nn.Sequential(nn.Conv1d(latent, ch, 1), nn.GELU(), nn.ConvTranspose1d(ch, 1, frame, stride=frame))
    def quantize(self, z):
        idx = torch.round((z + 1) / 2 * (self.levels - 1))
        zq = idx / (self.levels - 1) * 2 - 1
        return z + (zq - z).detach(), idx.long()      # straight-through estimator
    def forward(self, x):                              # x: (batch, samples), samples a multiple of frame
        zq, idx = self.quantize(self.enc(x.unsqueeze(1)))
        return self.dec(zq).squeeze(1), idx

model = TinyCodec().to(dev_name)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = torch.utils.data.DataLoader(s.SegmentDataset(dev, seconds=1.0, items=16 * 300), batch_size=16, num_workers=2)
t0, losses = time.time(), []
for step, xb in enumerate(loader):
    xb = xb.to(dev_name)
    y, _ = model(xb)
    loss = torch.mean((y - xb) ** 2)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if time.time() - t0 > 60:
        break
print('%d training steps in %.0f s; loss %.2e -> %.2e' % (step + 1, time.time() - t0, np.mean(losses[:10]), np.mean(losses[-10:])))
assert np.mean(losses[-10:]) < np.mean(losses[:10]), 'loss did not decrease'
results['training'] = 'ok'

In [ ]:
model = model.cpu().eval()
frame = model.frame
def codec(x):                                          # numpy in, (numpy out, indices)
    n = len(x) // frame * frame
    with torch.no_grad():
        y, idx = model(torch.from_numpy(x[:n]).unsqueeze(0))
    return y.squeeze(0).numpy(), idx.numpy()

p, q, r = [], [], []
for path, _ in test[:5]:
    x = s.read_audio(path)
    y, idx = codec(x)
    n_bytes = s.huffman_bytes(idx)                     # what would be transmitted, incl. code table
    r.append(s.bitrate_kbps(n_bytes, len(y) / 16000))
    p.append(s.pesq_wb(x[:len(y)], y)); q.append(s.stoi_score(x[:len(y)], y))
rtf = s.real_time_factor(lambda v: codec(v), s.read_audio(test[0][0]))
print('tiny codec: %.2f kbit/s (raw indices: %.1f kbit/s), algorithmic delay %.1f ms, PESQ %.2f, STOI %.3f, real-time factor %.4f on %s'
      % (np.mean(r), 8 * 4 * 100 / 1000, s.algorithmic_delay_ms(frame, 0), np.mean(p), np.mean(q), rtf['rtf'], rtf['cpu']))
assert 0 < np.mean(r) < 3.3 and rtf['rtf'] < 1
results['codec pipeline'] = 'ok'

In [ ]:
print(results)
assert len(results) == 4
print('Topic 15 pipeline check passed')